In [ ]:
# print("Hello world")

Hello world


In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [1]:
!pip install -U sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12942.28it/s]


In [5]:
!pip install minsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 19.1 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [minsearch]/3 [minsearch]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [6]:
from ingest import load_faq_data, build_index

In [13]:
import json

In [7]:
v1 = model.encode(q1)

In [5]:
v1.shape

(384,)

In [8]:
v2 = model.encode(q2)

In [9]:
d = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

In [10]:
dv = model.encode(d)

In [11]:
v1.dot(dv)

np.float32(0.39572883)

In [12]:
v2.dot(dv)

np.float32(0.46365508)

In [11]:
documents = load_faq_data()

In [12]:
print(documents[10])

{'id': '316180784f', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: How many hours per week am I expected to spend on this course?', 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}


In [15]:
json_object = json.dumps(documents[10], indent=4)

In [16]:
print(type(json_object))

<class 'str'>


In [17]:
print(json_object)

{
    "id": "316180784f",
    "course": "data-engineering-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: How many hours per week am I expected to spend on this course?",
    "answer": "It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer."
}


In [18]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [19]:
print(len(texts))

1368


In [20]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

100%|██████████| 28/28 [01:57<00:00,  4.19s/it]


In [23]:
v1.dot(vectors[10])

np.float32(0.31346333)

In [24]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [25]:
import numpy as np
X = np.array(vectors)

In [26]:
scores = X.dot(v1)

In [27]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(538), np.float32(0.831779))

In [28]:
documents[553]

{'id': 'a9353fadfe',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."}

In [29]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [30]:
scores[top5]

array([0.831779  , 0.6845695 , 0.61755615, 0.6088087 , 0.58479655],
      dtype=float32)

In [33]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.831779
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.6845695
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

0.61755615
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'se

In [34]:
top5 = np.argsort(-scores)[:5]

In [35]:
print(top5)

[538 925 643   2 503]
